In [3]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

# OpenPIV
from openpiv import windef
from openpiv import tools, scaling, validation, filters, preprocess
import openpiv.pyprocess as process
from openpiv import pyprocess

# General
import numpy as np
import tifffile as tif
import matplotlib.pyplot as plt
import pandas as pd
from roifile import ImagejRoi

import pathlib
from time import time
import warnings
%matplotlib qt

# Custom functions
from src.PIV import run_PIV_on_frames

In [20]:
# Load data 
vid_1_path = '/mnt/crunch/Clark/Fly_TFM/data/first/reference-left_the_frame/'
vid_2_path = '/mnt/crunch/Clark/Fly_TFM/data/first/reference-350/' 

n_points = 3969  # after loading once, or hardcode 3969
n_frames = 31

u_vid1_all = np.zeros((n_frames, n_points))
v_vid1_all = np.zeros((n_frames, n_points))
u_vid2_all = np.zeros((n_frames, n_points))
v_vid2_all = np.zeros((n_frames, n_points))

vid_1_reference_data = pd.read_csv(vid_1_path + 'PIVlab_0350.txt', skiprows=2)
vid_1_reference_u = vid_1_reference_data['u [px/frame]'].values
vid_1_reference_v = vid_1_reference_data['v [px/frame]'].values

for i, current_frame in enumerate(range(350, 381)):
    
    vid_1_data_path = vid_1_path + f'PIVlab_{current_frame:04d}.txt'
    vid_2_data_path = vid_2_path + f'PIVlab_{(current_frame - 349):04d}.txt'
    
    vid_1_data = pd.read_csv(vid_1_data_path, skiprows=2)
    vid_2_data = pd.read_csv(vid_2_data_path, skiprows=2)
    
    # print(vid_1_data_path)
    # print(vid_2_data_path)
    
    # Get values 
    u_vid1_all[i] = vid_1_data['u [px/frame]'].values - vid_1_reference_u
    v_vid1_all[i] = vid_1_data['v [px/frame]'].values - vid_1_reference_v
    u_vid2_all[i] = vid_2_data['u [px/frame]'].values
    v_vid2_all[i] = vid_2_data['v [px/frame]'].values
    
rms_per_point = np.sqrt(
    np.mean((u_vid2_all - u_vid1_all)**2 + (v_vid2_all - v_vid1_all)**2, axis=0)
)

In [28]:
def per_point_correlation(a, b):
    a_centered = a - a.mean(axis=0)
    b_centered = b - b.mean(axis=0)
    numerator = np.sum(a_centered * b_centered, axis=0)
    denominator = np.sqrt(np.sum(a_centered**2, axis=0) * np.sum(b_centered**2, axis=0))
    return numerator / denominator

corr_u = per_point_correlation(u_vid1_all, u_vid2_all)
corr_v = per_point_correlation(v_vid1_all, v_vid2_all)

In [26]:
print(np.shape(rms_per_point))

x_piv = vid_1_data['x [px]'].values
y_piv = vid_1_data['y [px]'].values

x_unique = np.unique(x_piv)
y_unique = np.unique(y_piv)
nx, ny = len(x_unique), len(y_unique)

col_idx = np.searchsorted(x_unique, x_piv)
row_idx = np.searchsorted(y_unique, y_piv)

rms_grid = np.zeros((ny, nx))
rms_grid[row_idx, col_idx] = rms_per_point

plt.figure(figsize=(10,10))
plt.imshow(rms_grid, cmap='jet', vmin=0, vmax=10, origin='lower', extent=[x_unique[0], x_unique[-1], y_unique[0], y_unique[-1]])
plt.colorbar(label='RMS error (px)')
plt.axis('equal')
plt.show()

(3969,)


In [36]:
corr_grid = np.zeros((ny, nx))
corr_grid[row_idx, col_idx] = corr_v  # or corr_v

plt.figure(figsize=(10,10))
plt.imshow(corr_grid, cmap='coolwarm', vmin=-1, vmax=1, origin='lower',
           extent=[x_unique[0], x_unique[-1], y_unique[0], y_unique[-1]])
plt.colorbar(label='correlation (v)')
plt.axis('equal')
plt.show()